In [ ]:
# Install Cadabra if necessary, and enable Cadabra input in this notebook.
try:
    import cadabra2
except ImportError:
    %pip install -q cadabra2
%load_ext cadabra2_jupyter

# Local supersymmetry of $N=1$, $d=4$ supergravity

This notebook shows that the action of $N=1$ supergravity in four dimensions is
invariant under local supersymmetry. It is a more complete version of the example
in section 3 of hep-th/0701238, and follows the discussion in sections 1.6 and 1.7 of
P. van Nieuwenhuizen, *Supergravity*, Phys. Rept. 68 (1981) 189 (referred to as vN below),
step by step.

We use the 1.5 order formalism: the spin connection $\omega_\mu{}^{mn}=\omega_\mu{}^{mn}(e,\psi)$
solves its own field equation, so its variation never has to be computed and we can set
$\delta\omega=0$. Only the vielbeine and gravitini which appear explicitly in the action are varied.
Conventions: $\kappa=1$, Pauli metric (so the flat metric is $\delta_{mn}$), and

$$
\delta e^m{}_\mu = \tfrac{1}{2}\bar\epsilon\Gamma^m\psi_\mu\,,\quad
\delta\psi_\mu = D_\mu\epsilon\,,\quad
D_\mu\epsilon = \partial_\mu\epsilon + \tfrac{1}{4}\omega_\mu{}^{mn}\Gamma_{mn}\epsilon\,.
$$

The covariant derivative $D_\mu$ contains only the spin connection, not a Christoffel
connection. 

## Declarations

Flat (tangent space) indices are Latin and curved indices are Greek. Curved indices have
`position=independent`, so Cadabra never raises or lowers them by itself, and every
change of index type has to go through an explicit vielbein $e^m{}_\mu$ or inverse
vielbein $e_m{}^\mu$. Flat indices are raised and lowered with $\delta_{mn}$. The range of the flat indices tells `join_gamma` that we are in four dimensions, so that products of gamma matrices never produce more than four antisymmetrised gammas.

In [ ]:
{m,n,p,q,r,s,t,u,v,w}::Indices(flat, position=fixed).
{m,n,p,q,r,s,t,u,v,w}::Integer(1..4).
{\mu,\nu,\rho,\sigma,\kappa,\lambda,\alpha,\beta}::Indices(curved, position=independent).
\delta{#}::KroneckerDelta.
\delta_{m n}::Metric.
\delta^{m n}::InverseMetric.
e^{m}_{\mu}::Vielbein.
e_{m}^{\mu}::InverseVielbein.

The gravitino $\psi_\mu$ and the supersymmetry parameter $\epsilon$ are anti-commuting Majorana spinors.
Index symmetries are attached to the exact index pattern they are declared with
(the indices are not `position=free`), so the curvature is declared and used as
$R_{\mu\nu}{}^{mn}$ throughout.

In [ ]:
\partial{#}::PartialDerivative.
D{#}::Derivative.
\bar{#}::DiracBar.
\Gamma{#}::GammaMatrix(metric=\delta).
{\epsilon, \psi{#}}::Spinor(dimension=4, type=Majorana).
{\epsilon, \psi{#}}::AntiCommuting.
\psi{#}::SelfAntiCommuting.
{\epsilon, \psi{#}}::SortOrder.
\Gamma{#}::Depends(\bar{#}).
\omega_{\mu}^{m n}::TableauSymmetry(shape={1,1}, indices={1,2}).
R_{\mu\nu}^{m n}::TableauSymmetry(shape={1,1}, indices={0,1}, shape={1,1}, indices={2,3}).
{\epsilon, \psi{#}, \omega_{\mu}^{m n}}::Depends(\partial{#}).

## Curvature conventions

Every sign in what follows depends on the normalisation of the commutator of two covariant
derivatives, so we check that first. With the covariant derivative given above, we want

$$
[D_\mu, D_\nu]\psi_\rho = \tfrac{1}{4} R_{\mu\nu}{}^{mn}\Gamma_{mn}\psi_\rho\,,
$$

which is vN 1.7 eq. (3) (with $\tfrac12\sigma^{ab} = \tfrac14\Gamma^{ab}$).
Note that $D_\mu$ acts only on the spinor index of $\psi_\rho$.

In [ ]:
covd:= D_{\mu}{A??} -> \partial_{\mu}{A??} + 1/4 \omega_{\mu}^{m n} \Gamma_{m n} A??;
comm:= D_{\mu}{D_{\nu}{\psi_{\rho}}} - D_{\nu}{D_{\mu}{\psi_{\rho}}};

In [ ]:
substitute(comm, covd, repeat=True)
distribute(comm)
product_rule(comm)
distribute(comm)
sort_product(comm)
join_gamma(comm)
distribute(comm)
unwrap(comm)
canonicalise(comm)
rename_dummies(comm);

This should equal $\tfrac14 R_{\mu\nu}{}^{mn}\Gamma_{mn}\psi_\rho$, with the curvature of the spin connection given by

In [ ]:
Rdef:= R_{\mu\nu}^{m n} -> \partial_{\mu}{\omega_{\nu}^{m n}} - \partial_{\nu}{\omega_{\mu}^{m n}}
         + \omega_{\mu}^{m q} \delta_{q p} \omega_{\nu}^{p n} - \omega_{\nu}^{m q} \delta_{q p} \omega_{\mu}^{p n};

In [ ]:
chk:= @(comm) - 1/4 R_{\mu\nu}^{m n} \Gamma_{m n} \psi_{\rho};
substitute(chk, Rdef)
distribute(chk)
sort_product(chk)
canonicalise(chk)
rename_dummies(chk)
collect_terms(chk);

## The action

The action is the sum of the Hilbert action $\mathcal{L}^{(2)}$ and the Rarita-Schwinger action
$\mathcal{L}^{(3/2)}$ (vN 1.7 eq. (1)). We keep the two parts separate, so we can follow
which terms in the variation cancel against which. In vN the Hilbert action is written
as $-\tfrac12 e\, e^{a\nu} e^{b\mu} R_{\mu\nu ab}$:

In [ ]:
L2:= -1/2 e e_{m}^{\nu} e_{n}^{\mu} R_{\mu\nu}^{m n};

vN writes the gravitino action as
$-\tfrac12\epsilon^{\mu\nu\rho\sigma}\bar\psi_\mu\gamma_5\gamma_\nu D_\rho\psi_\sigma$.
In four dimensions $\epsilon^{\mu\nu\rho\sigma}\gamma_5\gamma_\nu$ is proportional to
$e\,\Gamma^{\mu\rho\sigma}$, so we can write the action without $\gamma_5$ or an epsilon
tensor. The overall normalisation of this term relative to $\mathcal{L}^{(2)}$ is checked
later, when the Einstein tensor terms of the two parts have to cancel.
The curved indices on the gamma matrix are converted to flat ones, so that the vielbeine
appear explicitly and can be varied.

In [ ]:
L32:= -1/2 e \bar{\psi_{\mu}} \Gamma^{\mu\nu\rho} D_{\nu}{\psi_{\rho}};
rewrite_indices(_, $\Gamma^{m n p}$, $e_{m}^{\mu}$);

## Transformation rules

From $\delta e^m{}_\mu = \tfrac12\bar\epsilon\Gamma^m\psi_\mu$ and $\delta(e^m{}_\mu e_n{}^\mu)=0$
it follows that the inverse vielbein transforms as
$\delta e_m{}^\mu = -e_m{}^\nu\,(\delta e^n{}_\nu)\, e_n{}^\mu$. The variation of the
determinant is $\delta e = e\, e_m{}^\mu\delta e^m{}_\mu$. These are the explicit
vielbeine of vN 1.7 eq. (1). The gravitini are varied separately, later.

In [ ]:
dvb:= { e_{m}^{\mu} -> -1/2 \bar{\epsilon}\Gamma^{n}\psi_{\nu} e_{m}^{\nu} e_{n}^{\mu},
        e -> 1/2 e \bar{\epsilon}\Gamma^{m}\psi_{\mu} e_{m}^{\mu} };

## Variation of the vielbeine in the Hilbert action

We start with the variation of the explicit vielbeine in $\mathcal{L}^{(2)}$, vN 1.7 eq. (2).

In [ ]:
dL2:= @(L2);
vary(dL2, dvb)
distribute(dL2)
sort_product(dL2)
canonicalise(dL2)
rename_dummies(dL2)
collect_terms(dL2);

This should be equal to

$$
\delta\mathcal{L}^{(2)} = \frac{e}{2}\big(\bar\epsilon\gamma^\nu\psi^a\big)\, G_{\nu a}\,,\qquad
G_{\nu a} = R_{\nu a} - \tfrac12 e_{a\nu} R\,,\qquad R_{\nu a} = R_{\mu\nu ab}\, e^{b\mu}\,.
$$

Written out with all vielbeine explicit, this is

In [ ]:
eins:= 1/2 e \bar{\epsilon}\Gamma^{p}\psi_{\rho} e_{p}^{\nu} e_{q}^{\rho}
       ( R_{\mu\nu}^{q r} e_{r}^{\mu} - 1/2 e^{q}_{\nu} R_{\sigma\lambda}^{s t} e_{s}^{\lambda} e_{t}^{\sigma} );
distribute(eins)
substitute(eins, $e^{q}_{\nu} e_{q}^{\rho} -> \delta_{\nu}^{\rho}$)
eliminate_kronecker(eins)
sort_product(eins)
canonicalise(eins)
rename_dummies(eins);

The difference with the result of the variation vanishes:

In [ ]:
chk2:= @(dL2) - @(eins);
canonicalise(chk2)
rename_dummies(chk2)
collect_terms(chk2);

Only the antisymmetry of $R_{\mu\nu}{}^{mn}$ in both index pairs was used here. The pair
exchange symmetry does not hold, because the connection $\omega(e,\psi)$ has torsion.

## Variation of the gravitini

The gravitino appears twice in $\mathcal{L}^{(3/2)}$, so varying it gives two terms:
one with $\delta\psi_\rho = D_\rho\epsilon$ and one with
$\delta\bar\psi_\mu = \overline{D_\mu\epsilon}$. We will need to integrate by parts,
so this time we vary the action rather than the Lagrangian density.

In [ ]:
dpsi:= { \psi_{\mu} -> D_{\mu}{\epsilon} };
dL32:= \int{ @(L32) }{x};
vary(dL32, dpsi);

We handle the two terms one at a time, using `zoom` to hide the other one.
In the first term the two derivatives in $D_\nu D_\rho\epsilon$ are antisymmetrised by
$\Gamma^{mnp}e_n{}^\nu e_p{}^\rho$, so we can replace them by half the commutator.
$D_\mu$ contains no Christoffel connection, so no torsion term appears, and the curvature
check at the top gives $D_\nu D_\rho\epsilon \to \tfrac18 R_{\nu\rho}{}^{mn}\Gamma_{mn}\epsilon$
(vN 1.7 eq. (3)).

In [ ]:
zoom(dL32, $D_{\nu}{D_{\rho}{\epsilon}} Q??$);

In [ ]:
substitute(dL32, $D_{\nu}{D_{\rho}{\epsilon}} -> 1/8 R_{\nu\rho}^{m n} \Gamma_{m n} \epsilon$)
unzoom(dL32);

In the second term we use $\overline{D_\mu\epsilon} = D_\mu\bar\epsilon
= \partial_\mu\bar\epsilon - \tfrac14\bar\epsilon\,\omega_\mu{}^{mn}\Gamma_{mn}$ (vN 1.7 eq. (8)),
and integrate by parts to remove the derivative from $\bar\epsilon$. The total derivative is
the derivative of a Lorentz scalar, so $D_\mu=\partial_\mu$ on it and we may drop it.

In [ ]:
zoom(dL32, $\bar{D_{\mu}{\epsilon}} Q??$);

In [ ]:
substitute(dL32, $\bar{D_{\mu}{\epsilon}} -> D_{\mu}{\bar{\epsilon}}$)
integrate_by_parts(dL32, $\bar{\epsilon}$);

The derivative $D_\mu$ now acts on the determinant, the inverse vielbeine, the gamma matrix
and $D_\nu\psi_\rho$. A gamma matrix with flat indices is covariantly constant. The derivatives
of the determinant and of the inverse vielbeine can all be written in terms of $D_\mu e^q{}_\lambda$.
This is not zero, because $D_\mu$ contains only the spin connection (see vN 1.4 below eq. (3)):

$$
D_\mu e = e\, e_q{}^\lambda D_\mu e^q{}_\lambda\,,\qquad
D_\mu e_m{}^\nu = - e_m{}^\lambda (D_\mu e^q{}_\lambda)\, e_q{}^\nu\,.
$$

The two covariant derivatives on the gravitino are antisymmetrised again, so they also
become a curvature, $D_\mu D_\nu\psi_\rho \to \tfrac18 R_{\mu\nu}{}^{mn}\Gamma_{mn}\psi_\rho$.

In [ ]:
product_rule(dL32)
distribute(dL32)
substitute(dL32, $D_{\mu}{\Gamma^{m n p}} -> 0$)
substitute(dL32, $D_{\mu}{e} -> e e_{q}^{\lambda} D_{\mu}{e^{q}_{\lambda}}$)
substitute(dL32, $D_{\mu}{e_{m}^{\nu}} -> - e_{m}^{\lambda} D_{\mu}{e^{q}_{\lambda}} e_{q}^{\nu}$)
substitute(dL32, $D_{\mu}{D_{\nu}{\psi_{\rho}}} -> 1/8 R_{\mu\nu}^{m n} \Gamma_{m n} \psi_{\rho}$)
distribute(dL32);

In [ ]:
unzoom(dL32);

## The curvature terms

There are now two kinds of terms: curvature terms, and terms containing $D_\mu e^q{}_\lambda$
(the terms of vN 1.7 eq. (9)). We zoom in on the curvature terms and simplify the gamma
matrix algebra. The terms with five gamma matrices vanish in four dimensions. To compare
the terms coming from the two gravitino variations, all spinor bilinears are written with
$\bar\epsilon$ on the left, using the Majorana flip relations (vN 1.7 eq. (5)).
The terms with three gamma matrices then cancel.

In [ ]:
zoom(dL32, $R_{\mu\nu}^{m n} Q??$);

In [ ]:
sort_product(dL32)
join_gamma(dL32)
distribute(dL32)
eliminate_kronecker(dL32)
sort_spinors(dL32)
sort_product(dL32)
canonicalise(dL32)
rename_dummies(dL32)
collect_terms(dL32);

The visible terms are $-\frac{e}{2}(\bar\psi^\mu\gamma^a\epsilon)G_{a\mu}$, vN 1.7 eq. (7).
They should cancel exactly against the variation of the vielbeine in the Hilbert action.
To check this we remove the integral sign again (it has done its job) and add `dL2`.

In [ ]:
unzoom(dL32)
substitute(dL32, $\int{A??}{x} -> A??$);

In [ ]:
rest:= @(dL32) + @(dL2);
sort_product(rest)
canonicalise(rest)
rename_dummies(rest)
collect_terms(rest);

All curvature terms have cancelled. Only the terms with derivatives of the vielbein remain.
This also confirms that $\mathcal{L}^{(3/2)}$ has the right normalisation and sign relative
to the Hilbert action, so the $\Gamma^{\mu\nu\rho}$ form is equivalent to vN's
$\epsilon^{\mu\nu\rho\sigma}\gamma_5\gamma_\nu$ form.

To finish, we have to show that these remaining terms cancel against the variation of the
explicit vielbeine in $\mathcal{L}^{(3/2)}$ (vN 1.7 eq. (10)). For that we need the torsion
equation for $D_{[\mu}e^a{}_{\nu]}$ and a Fierz rearrangement; that comes next.

## Variation of the explicit vielbeine

$\mathcal{L}^{(3/2)}$ also depends on the vielbein, through $e$ and through the inverse
vielbeine which convert $\Gamma^{\mu\nu\rho}$ to flat indices. Varying these gives terms
of order $\psi^3\epsilon$ (vN 1.7 eq. (10)).

In [ ]:
dL32e:= @(L32);
vary(dL32e, dvb)
distribute(dL32e)
sort_product(dL32e)
canonicalise(dL32e)
rename_dummies(dL32e);

## Torsion

In the 1.5 order formalism $\omega$ satisfies its own field equation. That makes the
antisymmetric part of $D_\mu e^a{}_\nu$ equal to the gravitino torsion (vN 1.7 eq. (11)).
The symmetric part is the Christoffel symbol (recall that $D_\mu$ contains only the spin
connection), which we call $C_{\mu\nu}{}^q$ so that it sorts before $D$:

$$
D_\mu e^q{}_\nu = C_{\mu\nu}{}^q + \tfrac14\bar\psi_\mu\Gamma^q\psi_\nu\,,\qquad
C_{\mu\nu}{}^q = C_{\nu\mu}{}^q\,.
$$


In [ ]:
C_{\mu\nu}^{m}::TableauSymmetry(shape={2}, indices={0,1}).
tors:= D_{\mu}{e^{q}_{\nu}} -> C_{\mu\nu}^{q} + 1/4 \bar{\psi_{\mu}}\Gamma^{q}\psi_{\nu};

In [ ]:
substitute(rest, tors)
distribute(rest)
sort_product(rest)
canonicalise(rest)
rename_dummies(rest);

The Christoffel terms have cancelled, as they should: the variation of $\mathcal{L}^{(3/2)}$
has to be covariant. Adding the variation of the explicit vielbeine, everything left is
of the form $\bar\epsilon\psi\,\bar\psi D\psi$ or $\bar\psi\psi\,\bar\epsilon D\psi$.
To compare these we first move the bilinears containing $\bar\epsilon$ to the front.

In [ ]:
tot:= @(rest) + @(dL32e);
substitute(tot, $\bar{\psi_{\mu}} A?? B?? \bar{\epsilon} C?? D?? -> \bar{\epsilon} C?? D?? \bar{\psi_{\mu}} A?? B??$)
sort_product(tot)
canonicalise(tot)
rename_dummies(tot);

## Fierz rearrangement

Next we Fierz all terms so that $\bar\epsilon$ and $D\psi$ sit in the same bilinear
(vN 1.7 eq. (12)). For this, `fierz` needs to know that the covariant derivative of the
gravitino is a spinor. The result will not look pretty, partially because we are not using
a $\Gamma^5$ notation.

In [ ]:
D_{\mu}{\psi_{\nu}}::Spinor(dimension=4, type=Majorana).
fierz(tot, $\epsilon, D_{\mu}{\psi_{\sigma}}, \psi_{\rho}, \psi_{\nu}$);

All curved indices are now contracted with inverse vielbeine. We convert to flat indices,
writing $\chi_{mn} = e_m{}^\mu e_n{}^\nu D_\mu\psi_\nu$, and then simplify the products of
gamma matrices.

In [ ]:
\chi{#}::Spinor(dimension=4, type=Majorana).
{\epsilon, \psi{#}, \chi{#}}::AntiCommuting.
flat:= { D_{\mu}{\psi_{\nu}} e_{m}^{\mu} e_{n}^{\nu} -> \chi_{m n},
         \psi_{\mu} e_{m}^{\mu} -> \psi_{m},
         \bar{\psi_{\mu}} e_{m}^{\mu} -> \bar{\psi_{m}} };
substitute(tot, flat, repeat=True);

In [ ]:
converge(tot):
    join_gamma(tot)
    distribute(tot)
    eliminate_kronecker(tot)

converge(tot):
    sort_product(tot)
    canonicalise(tot)
    rename_dummies(tot)
    collect_terms(tot)
;

## The final cancellation

The remaining terms all contain $\Gamma^{mnpq}$ and five different flat indices.
In four dimensions they vanish because antisymmetrising over five indices gives zero
(a Schouten identity). We make this explicit the way vN does, with the epsilon tensor and
$\gamma_5 = \Gamma_1\Gamma_2\Gamma_3\Gamma_4$. With our Euclidean metric,

$$
\Gamma^{mnpq} = \varepsilon^{mnpq}\gamma_5\,,\quad
\Gamma_{mn} = -\tfrac12\varepsilon_{mnpq}\Gamma^{pq}\gamma_5\,.
$$

We first lower all indices on the $\Gamma_{mn}$, so that every epsilon has all its lower
indices together. Then the products of two epsilon tensors become Kronecker deltas, and
everything cancels.

In [ ]:
\varepsilon{#}::EpsilonTensor(delta=\delta).
\gamma_{5}::ImplicitIndex.
substitute(tot, $\Gamma_{m}^{n} -> \Gamma_{m p} \delta^{p n}$)
substitute(tot, $\Gamma^{m n p q} -> \varepsilon^{m n p q} \gamma_{5}$)
substitute(tot, $\Gamma_{m n} -> -1/2 \varepsilon_{m n p q} \Gamma^{p q} \gamma_{5}$);

In [ ]:
epsilon_to_delta(tot)
expand_delta(tot)
distribute(tot)
eliminate_kronecker(tot);
converge(tot):
    sort_product(tot)
    canonicalise(tot)
    rename_dummies(tot)
    collect_terms(tot)
;

This completes the proof that the action $\mathcal{L}^{(2)} + \mathcal{L}^{(3/2)}$ is invariant
under the local supersymmetry transformations
$\delta e^m{}_\mu = \tfrac12\bar\epsilon\Gamma^m\psi_\mu$, $\delta\psi_\mu = D_\mu\epsilon$.